# PCB part detector: training on tiles, on a free Kaggle GPU

**Before running:**
1. *Add Input* → your `pcb_tiles` dataset (the zip written by `python train.py tiles RF100 pcb_tiles`).
2. *Session options* → Accelerator **GPU T4 x2** (or P100).
3. **Internet:** turn it **on** in *Session options* (needs a phone-verified Kaggle account).
   If you can't, also *Add Input* → a dataset made from `kaggle_offline.zip`
   (training library wheels + starting weights); the first cell then installs offline.
4. *Save Version* → *Save & Run All* (the output is kept even if the browser closes). About 1.5 h. Then download `detector_export.zip` from the notebook's *Output* tab.

Model: YOLOv8**s**, 1024 px tiles cut from photos at 2048 px.
The 22 test boards are never used for training or model selection.

In [ ]:
import glob, json, os, shutil, socket, subprocess, sys, time
import yaml
from pathlib import Path

def online():
    try:
        socket.create_connection(('pypi.org', 443), timeout=5).close()
        return True
    except OSError:
        return False

ONLINE = online()
offline_dir = [Path(p).parent for p in glob.glob('/kaggle/input/**/yolov8s.pt', recursive=True)]
wheels = glob.glob('/kaggle/input/**/wheels/*.whl', recursive=True)
if ONLINE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.3.40', 'onnx', 'onnxslim'], check=True)
elif wheels:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', *wheels], check=True)
else:
    raise SystemExit('Internet is OFF and no offline bundle found. Either turn Internet on '
                     '(Session options) or Add Input -> the kaggle_offline dataset, then Run All again.')

os.chdir('/kaggle/working')
if offline_dir:        # starting weights + AMP-check model + plot font, so nothing needs downloading
    for f in ('yolov8s.pt', 'yolo11n.pt'):
        if (offline_dir[0] / f).exists():
            shutil.copy(offline_dir[0] / f, f)
    font = offline_dir[0] / 'Arial.ttf'
    if font.exists():
        cfg = Path.home() / '.config' / 'Ultralytics'; cfg.mkdir(parents=True, exist_ok=True)
        shutil.copy(font, cfg / 'Arial.ttf')

import torch, ultralytics
from ultralytics import settings
# Kaggle pre-installs ray, wandb, mlflow, ... and ultralytics hooks into whichever it finds.
# The Ray Tune hook crashes with Kaggle's newer ray, and wandb can stop to ask for a login,
# so switch every third-party integration off (we only need the local results files).
settings.update({k: False for k in ('raytune', 'wandb', 'mlflow', 'comet', 'clearml', 'dvc',
                                    'neptune', 'tensorboard', 'hub') if k in settings})
os.environ['WANDB_DISABLED'] = 'true'
print('online:', ONLINE, '| ultralytics', ultralytics.__version__, '| torch', torch.__version__,
      '| GPUs:', torch.cuda.device_count())
assert torch.cuda.is_available(), 'No GPU: set Accelerator to GPU T4 x2 in Session options'

In [ ]:
# find the uploaded dataset (folder that holds data.yaml + summary.json)
cands = [Path(p).parent for p in glob.glob('/kaggle/input/**/summary.json', recursive=True)]
assert cands, 'Upload pcb_tiles.zip as a Kaggle dataset first'
DATA = cands[0]
print(DATA, os.listdir(DATA))
summary = json.loads((DATA / 'summary.json').read_text())
print(json.dumps(summary, indent=1)[:800])
cfg = yaml.safe_load((DATA / 'data.yaml').read_text())
cfg['path'] = str(DATA)
Path('/kaggle/working/data.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
names = [cfg['names'][i] for i in sorted(cfg['names'])]
names

In [ ]:
from ultralytics import YOLO
IMGSZ = summary['tile']                     # 1024
t0 = time.time()
model = YOLO('yolov8s.pt')                  # COCO-pretrained start point
model.train(data='/kaggle/working/data.yaml', epochs=100, imgsz=IMGSZ, batch=16, device=0, workers=4,
            patience=25, time=2.5, plots=True, project='/kaggle/working/runs', name='parts', exist_ok=True,
            seed=0, deterministic=True, fliplr=0.5, flipud=0.5, degrees=5, scale=0.3, close_mosaic=10,
            max_det=1000, cache='ram')
train_minutes = (time.time() - t0) / 60
print(f'{train_minutes:.0f} min')

In [ ]:
best = YOLO('/kaggle/working/runs/parts/weights/best.pt')
m = best.val(data='/kaggle/working/data.yaml', split='test', imgsz=IMGSZ, batch=16, device=0,
             project='/kaggle/working/runs', name='test', exist_ok=True, max_det=1000)
per_class = {names[c]: round(float(m.box.ap50[i]), 3) for i, c in enumerate(m.box.ap_class_index)}
print(round(float(m.box.map50), 3), per_class)

In [ ]:
try:   # needs the onnx package (installed when online); otherwise best.pt is exported to ONNX locally
    onnx = Path(best.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=ONLINE, dynamic=False))
except Exception as e:
    print('ONNX export skipped (will be done locally from best.pt):', e)
    onnx = None
b = summary['boards']
meta = {
    'names': names, 'imgsz': IMGSZ,
    'metrics': {'map50': round(float(m.box.map50), 3), 'map50_95': round(float(m.box.map), 3),
                'precision': round(float(m.box.mp), 3), 'recall': round(float(m.box.mr), 3)},
    'per_class_ap50': per_class,
    'training': {'base': 'yolov8s.pt', 'epochs_requested': 100, 'imgsz': IMGSZ,
                 'minutes': round(train_minutes, 1), 'smoke_test': False, 'device': 'kaggle-gpu'},
    'dataset': (f"Trained on {b['train']} boards, validated on {b['valid']}, tested on {b['test']} unseen boards "
                f"(as {summary['tiles']['train']} training tiles). Source: {summary['source']}."),
    'tiling': {'long_side': summary['long_side'], 'tile': summary['tile'], 'overlap': summary['overlap'],
               'metrics_are': 'per tile (see tiling_eval.json for whole boards)'},
}
out = Path('/kaggle/working/detector_export'); out.mkdir(exist_ok=True)
if onnx: shutil.copy(onnx, out / 'parts.onnx')
shutil.copy('/kaggle/working/runs/parts/weights/best.pt', out / 'best.pt')
(out / 'parts.json').write_text(json.dumps(meta, indent=2))
for f in ['results.csv', 'results.png', 'confusion_matrix_normalized.png', 'PR_curve.png']:
    p = Path('/kaggle/working/runs/parts') / f
    if p.exists(): shutil.copy(p, out / f)
shutil.make_archive('/kaggle/working/detector_export', 'zip', out)
print(json.dumps(meta, indent=2))